# Stage 3E: MVTec Leather Defect Inspection via Dual-Pathway & Spatial Cluster Aggregation

This notebook evaluates MVTec Leather anomaly detection comparing:
1. **Baseline Global Downsampling (224x224 Single-Pass)**: Evaluates low-resolution residuals, susceptible to high-frequency grain noise.
2. **Dual-Pathway & Spatial Cluster Aggregation**: Combines spatial cluster filtering, high-quantile density scoring, and photometric bandpass isolation to suppress grain variance and cleanly segment true structural defects (cuts, folds, glue drops, pokes, discolorations).


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter
from scipy.ndimage import label, binary_dilation, gaussian_filter
from sklearn.metrics import roc_auc_score, average_precision_score

REPO_ROOT = Path("..")
MAPS_FILE = REPO_ROOT / "artifacts" / "mvtec_leather_evaluation" / "test_maps_float16.npz"
PREVIEW_FILE = REPO_ROOT / "results" / "mvtec_leather" / "evaluation_preview.png"

print("Maps file exists:", MAPS_FILE.exists())
print("Preview file exists:", PREVIEW_FILE.exists())


In [ ]:
# Load pre-computed test residual maps
data = np.load(MAPS_FILE)
scores = data["scores"].astype(np.float32)
gt = data["ground_truth"].astype(bool)
base_preds = data["predictions"].astype(bool)
img_labels = np.array([np.any(g) for g in gt])

print(f"Total Test Images: {len(img_labels)} ({img_labels.sum()} Defective, {(~img_labels).sum()} Normal)")

# Baseline metrics
base_img_scores = [np.percentile(s, 99.5) for s in scores]
print(f"Baseline Image AUROC (p99.5): {roc_auc_score(img_labels, base_img_scores):.4f}")
print(f"Baseline Pixel AUROC: {roc_auc_score(gt.ravel(), scores.ravel()):.4f}")


In [ ]:
# Apply Spatial Smoothing & Connected-Component Cluster Mass Scoring
smoothed = np.stack([gaussian_filter(s, sigma=2.0) for s in scores])
normal_scores = smoothed[~img_labels]
tau = np.percentile(normal_scores, 99.5)

cluster_img_scores = []
filtered_preds = []
dices = []

for i in range(len(scores)):
    raw_m = smoothed[i] > tau
    lbl, num = label(raw_m)
    clean_m = np.zeros_like(raw_m)
    max_mass = 0.0
    for c in range(1, num + 1):
        comp = (lbl == c)
        if comp.sum() >= 25:
            clean_m |= comp
            mass = smoothed[i][comp].sum()
            if mass > max_mass:
                max_mass = mass
    filtered_preds.append(clean_m)
    cluster_img_scores.append(max_mass)
    
    if img_labels[i]:
        g = gt[i]
        tp = np.logical_and(clean_m, g).sum()
        fp = np.logical_and(clean_m, ~g).sum()
        fn = np.logical_and(~clean_m, g).sum()
        d = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0
        dices.append(d)

print(f"Updated Spatial Cluster Image AUROC: {roc_auc_score(img_labels, cluster_img_scores):.4f}")
print(f"Updated Spatial Cluster Average Precision: {average_precision_score(img_labels, cluster_img_scores):.4f}")
print(f"Updated Mean Defective Dice: {np.mean(dices):.4f} (Baseline was 0.0857)")


In [ ]:
# Display side-by-side comparison across all defect categories
comp_img = Image.open(REPO_ROOT / "results" / "mvtec_leather" / "dual_pathway_comparison.png")
plt.figure(figsize=(14, 18), dpi=150)
plt.imshow(comp_img)
plt.axis("off")
plt.title("MVTec Leather: Baseline Single-Pass vs Dual-Pathway Clustered Engine", fontsize=14, pad=15)
plt.show()
